In [ ]:
!pip install vllm fastembed-gpu pyngrok nest-asyncio sentence-transformers fastapi uvicorn -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117

In [ ]:
"""
@file colab_gpu_worker.py
@description Sovereign Edge Node. Features a Two-Layer Jupyter OS Compatibility Patch for vLLM.
@layer Core Logic
"""
import sys
import io
import contextlib
import os

# ── SOTA FIX: Two-Layer Jupyter OS Compatibility Patch ──
try: sys.stdout.fileno()
except io.UnsupportedOperation: sys.stdout.fileno = lambda: 1

try: sys.stderr.fileno()
except io.UnsupportedOperation: sys.stderr.fileno = lambda: 2

try:
    import vllm.utils.system_utils
    vllm.utils.system_utils.suppress_stdout = contextlib.nullcontext
except ImportError:
    pass

import nest_asyncio
import uvicorn
import logging
import traceback
import time
import torch
import uuid
from fastapi import FastAPI, HTTPException, Security, Depends
from fastapi.security.api_key import APIKeyHeader
from pydantic import BaseModel
from sentence_transformers import CrossEncoder
from fastembed import LateInteractionTextEmbedding
from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s")
logger = logging.getLogger("Sovereign-Node")

TUNNEL_API_KEY = "omni_colab_secret_123"
api_key_header = APIKeyHeader(name="X-API-Key", auto_error=True)

app = FastAPI(title="ATLAS Sovereign Edge Node")

def verify_api_key(api_key: str = Security(api_key_header)):
    if api_key != TUNNEL_API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")

# ── Model Pre-Warming & Memory Fencing ───────────────────────────────────
# Reverted to the stable baseline: 55% utilization easily holds an 8k context window
# (approx 30k KV cache tokens) while leaving ~6.5GB for FastEmbed and the Reranker.
logger.info("Initializing vLLM (Qwen-2.5-7B-AWQ)... Fencing VRAM to 55%.")
engine_args = AsyncEngineArgs(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    quantization="awq",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.55,
    max_model_len=8192,
    enforce_eager=True
)
llm_engine = AsyncLLMEngine.from_engine_args(engine_args)

logger.info("Loading Jina-ColBERT-v2 via FastEmbed...")
embedder = LateInteractionTextEmbedding("jinaai/jina-colbert-v2", providers=["CUDAExecutionProvider"])

logger.info("Loading MS-MARCO Reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device='cuda', max_length=512)
logger.info("All Sovereign Node Systems Online.")

# ── Request Models ───────────────────────────────────────────────────────
class EmbedRequest(BaseModel): texts: list[str]
class RerankRequest(BaseModel): query: str; chunks: list[str]
class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 2048
    temperature: float = 0.0

# ── Endpoints ────────────────────────────────────────────────────────────
@app.get("/health", dependencies=[Depends(verify_api_key)])
async def health_check():
    return {"status": "online", "gpu_available": torch.cuda.is_available()}

@app.post("/generate", dependencies=[Depends(verify_api_key)])
async def generate(req: GenerateRequest):
    request_id = str(uuid.uuid4())
    logger.info(f"[LLM] Extraction requested. Prompt length: {len(req.prompt)} chars.")

    try:
        t0 = time.time()
        sampling_params = SamplingParams(temperature=req.temperature, max_tokens=req.max_tokens)

        results_generator = llm_engine.generate(req.prompt, sampling_params, request_id)
        final_output = None
        async for request_output in results_generator:
            final_output = request_output

        text = final_output.outputs[0].text
        logger.info(f"[LLM] ✅ Success: Generated {len(text)} chars in {time.time() - t0:.2f}s.")
        return {"text": text}
    except Exception as e:
        logger.error(traceback.format_exc())
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/embed", dependencies=[Depends(verify_api_key)])
async def embed(req: EmbedRequest):
    safe_texts = [t for t in req.texts if t and t.strip()]
    if not safe_texts: return {"vectors": []}
    try:
        embeddings_iter = embedder.embed(safe_texts)
        vectors = [matrix.tolist() for matrix in embeddings_iter]
        return {"vectors": vectors}
    except Exception as e:
        logger.error(traceback.format_exc())
        torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/rerank", dependencies=[Depends(verify_api_key)])
async def rerank(req: RerankRequest):
    safe_chunks = [c for c in req.chunks if c and c.strip()]
    if not safe_chunks: return {"scores": []}
    try:
        pairs = [[req.query, chunk] for chunk in safe_chunks]
        with torch.inference_mode():
            tensors = reranker.predict(pairs, convert_to_tensor=True)
            scores = tensors.float().cpu().numpy().tolist()
        return {"scores": scores}
    except Exception as e:
        logger.error(traceback.format_exc())
        torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=str(e))

# ── Execution ────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print(f"\n{'='*70}\n🚀 STARTING CLOUDFLARE TUNNEL (Wait 5-10 seconds...)\n{'='*70}\n")

    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
    os.system("chmod +x cloudflared")
    os.system("nohup ./cloudflared tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &")

    time.sleep(4)
    url_log = os.popen(r"grep -o 'https://.*\.trycloudflare.com' cloudflared.log").read().strip()
    if url_log:
        print(f"\n🔥 CLOUDFLARE TUNNEL LIVE AT: {url_log}\n🔑 API Key: {TUNNEL_API_KEY}\n")
    else:
        print("\n⏳ Tunnel booting... Run `!cat cloudflared.log` in a new cell if URL doesn't appear.\n")

    nest_asyncio.apply()
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    import asyncio
    asyncio.get_event_loop().run_until_complete(server.serve())

INFO 04-23 00:41:44 [model.py:549] Resolved architecture: Qwen2ForCausalLM
INFO 04-23 00:41:44 [model.py:1678] Using max model len 8192
INFO 04-23 00:41:44 [awq_marlin.py:249] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

WARNING 04-23 00:41:45 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-23 00:41:45 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 04-23 00:41:45 [vllm.py:1025] Cudagraph is disabled under eager mode
WARNING 04-23 00:41:46 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


🚀 STARTING CLOUDFLARE TUNNEL (Wait 5-10 seconds...)


🔥 CLOUDFLARE TUNNEL LIVE AT: https://appeared-thinkpad-journals-differential.trycloudflare.com
🔑 API Key: omni_colab_secret_123

WARNING 04-23 00:45:52 [input_processor.py:235] Passing raw prompts to InputProcessor is deprecated and will be removed in v0.18. You should instead pass the outputs of Renderer.render_cmpl() or Renderer.render_chat().


ERROR:Sovereign-Node:Traceback (most recent call last):
  File "/tmp/ipykernel_19823/4262028998.py", line 95, in generate
    async for request_output in results_generator:
  File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/async_llm.py", line 563, in generate
    q = await self.add_request(
        ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/async_llm.py", line 356, in add_request
    request = self.input_processor.process_inputs(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/input_processor.py", line 244, in process_inputs
    processed_inputs = self.input_preprocessor.preprocess(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/vllm/inputs/preprocess.py", line 288, in preprocess
    return self._process_decoder_only_prompt(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/di

KeyboardInterrupt: 